# htcie Benchmark — Correlation Output vs. Validated Reference Data

This notebook verifies that every htcie correlation produces Nusselt numbers within
expected ranges sourced from:

> Incropera, F.P. et al., *Fundamentals of Heat and Mass Transfer*, 7th Edition.

Three domains are covered:
- **Internal forced convection** in circular tubes
- **External forced convection** over cylinders and flat plates
- **Tube banks** (staggered arrangement)

Each case shows every applicable correlation alongside its computed Nu, the
textbook-derived expected range, and a pass/fail indicator.

In [1]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots

from htcie.core.loader import build_registry
from htcie.core.applicability import ApplicabilityEngine
from htcie.core.evaluator import EvaluationEngine
from htcie.core.state import (
    EngineeringState,
    FluidProperties,
    Geometry,
    BoundaryConditions,
    FlowState,
)

registry = build_registry()
applicability = ApplicabilityEngine()
evaluator = EvaluationEngine()

print(f"Registry loaded: {len(registry.all())} correlations")
print(f"Families: {registry.families()}")

Registry loaded: 13 correlations
Families: ['convection_external', 'convection_internal', 'tube_banks']


## Helper: run a benchmark case

The helper runs the applicability filter and evaluator for a given `EngineeringState`,
then checks each applicable correlation's Nu against the expected range.

In [2]:
def run_case(label, state, nu_lo, nu_hi):
    """Return list of dicts: {case, method, title, nu, nu_lo, nu_hi, pass_} for all applicable methods."""
    app = applicability.evaluate(state, registry.all())
    rows = []
    for method in app.applicable:
        result = evaluator.evaluate(state, method)
        nu = result.value
        rows.append(
            {
                "case": label,
                "method": method.key,
                "title": method.title,
                "nu": round(nu, 2),
                "nu_lo": nu_lo,
                "nu_hi": nu_hi,
                "pass_": nu_lo <= nu <= nu_hi,
            }
        )
    return rows


def print_case(label, rows):
    print(f"--- {label} ---")
    print(f"{' Method':<40} {'Nu':>8}  {'Expected Range':>20}  {'Pass?':>6}")
    print("-" * 82)
    for r in rows:
        lo, hi = r["nu_lo"], r["nu_hi"]
        print(
            f"{r['method']:<40} {r['nu']:>8.2f}  [{lo:>6.1f}, {hi:>6.1f}]  {'\u2713' if r['pass_'] else '\u2717':>6}"
        )
    print()

## Section 1 — Internal Forced Convection

Two fluids through smooth circular tubes, fully turbulent regime.

| Parameter | Water 60°C | Air 300K |
|-----------|-----------|----------|
| ρ [kg/m³] | 983.2 | 1.177 |
| μ [Pa·s] | 4.67×10⁻⁴ | 1.846×10⁻⁵ |
| k [W/m·K] | 0.654 | 0.0263 |
| cp [J/kg·K] | 4185 | 1007 |
| D [mm] | 25 | 20 |
| U [m/s] | 1.0 | 10.0 |
| Re | ≈52 634 | ≈12 758 |
| Pr | ≈2.99 | ≈0.707 |

Reference: Incropera 7th ed., §8.4, Example 8.3.

In [3]:
water_60c = EngineeringState(
    fluid=FluidProperties(
        density=983.2,
        viscosity=4.67e-4,
        thermal_conductivity=0.654,
        heat_capacity=4185.0,
    ),
    geometry=Geometry(
        geometry_type="circular_tube",
        characteristic_length=0.025,
        hydraulic_diameter=0.025,
    ),
    boundary=BoundaryConditions(boundary_type="constant_wall_temperature"),
    flow=FlowState(velocity=1.0),
)
print(f"Water 60°C → Re={water_60c.reynolds:.0f}, Pr={water_60c.prandtl:.3f}")
rows_water = run_case("water_60C", water_60c, nu_lo=200, nu_hi=300)
print_case("Water 60°C — Circular Tube", rows_water)

air_300k_tube = EngineeringState(
    fluid=FluidProperties(
        density=1.177,
        viscosity=1.846e-5,
        thermal_conductivity=0.0263,
        heat_capacity=1007.0,
    ),
    geometry=Geometry(
        geometry_type="circular_tube",
        characteristic_length=0.020,
        hydraulic_diameter=0.020,
    ),
    boundary=BoundaryConditions(boundary_type="constant_wall_temperature"),
    flow=FlowState(velocity=10.0),
)
print(
    f"Air 300K tube → Re={air_300k_tube.reynolds:.0f}, Pr={air_300k_tube.prandtl:.3f}"
)
rows_air_tube = run_case("air_300K_tube", air_300k_tube, nu_lo=30, nu_hi=55)
print_case("Air 300K — Circular Tube", rows_air_tube)

Water 60°C → Re=52634, Pr=2.988
--- Water 60°C — Circular Tube ---
 Method                                        Nu        Expected Range   Pass?
----------------------------------------------------------------------------------
internal.dittus_boelter                    213.26  [ 200.0,  300.0]       ✓
internal.gnielinski                        235.77  [ 200.0,  300.0]       ✓
internal.petukhov                          230.80  [ 200.0,  300.0]       ✓
internal.sieder_tate                       232.73  [ 200.0,  300.0]       ✓

Air 300K tube → Re=12752, Pr=0.707
--- Air 300K — Circular Tube ---
 Method                                        Nu        Expected Range   Pass?
----------------------------------------------------------------------------------
internal.dittus_boelter                     38.54  [  30.0,   55.0]       ✓
internal.gnielinski                         36.35  [  30.0,   55.0]       ✓
internal.petukhov                           36.41  [  30.0,   55.0]       ✓
intern

## Section 2 — External Forced Convection

### 2a — Cylinder in crossflow
Air at 300K, D=20mm, U=10 m/s (Re≈12 758).
Applicable: Churchill-Bernstein, Hilpert, Žukauskas. Reference: Incropera 7th §7.4.

### 2b — Flat plate (laminar)
Air at 300K, L=0.3m, U=5 m/s (Re≈95 800, laminar Re < 5×10⁵).
Applicable: Pohlhausen. Reference: Incropera 7th §7.2.

In [4]:
air_props = FluidProperties(
    density=1.177, viscosity=1.846e-5, thermal_conductivity=0.0263, heat_capacity=1007.0
)

air_cylinder = EngineeringState(
    fluid=air_props,
    geometry=Geometry(geometry_type="cylinder_crossflow", characteristic_length=0.020),
    boundary=BoundaryConditions(boundary_type="constant_wall_temperature"),
    flow=FlowState(velocity=10.0),
)
print(f"Air cylinder → Re={air_cylinder.reynolds:.0f}, Pr={air_cylinder.prandtl:.3f}")
rows_cylinder = run_case("air_cylinder", air_cylinder, nu_lo=45, nu_hi=80)
print_case("Air 300K — Cylinder in Crossflow", rows_cylinder)

air_plate = EngineeringState(
    fluid=air_props,
    geometry=Geometry(geometry_type="flat_plate", characteristic_length=0.3),
    boundary=BoundaryConditions(boundary_type="constant_wall_temperature"),
    flow=FlowState(velocity=5.0),
)
print(f"Air flat plate → Re={air_plate.reynolds:.0f}, Pr={air_plate.prandtl:.3f}")
rows_plate = run_case("air_flat_plate", air_plate, nu_lo=150, nu_hi=220)
print_case("Air 300K — Flat Plate (Laminar)", rows_plate)

Air cylinder → Re=12752, Pr=0.707
--- Air 300K — Cylinder in Crossflow ---
 Method                                        Nu        Expected Range   Pass?
----------------------------------------------------------------------------------
external.churchill_bernstein                61.28  [  45.0,   80.0]       ✓
external.hilpert                            59.23  [  45.0,   80.0]       ✓
external.zukauskas_cylinder                 66.46  [  45.0,   80.0]       ✓

Air flat plate → Re=95639, Pr=0.707
--- Air 300K — Flat Plate (Laminar) ---
 Method                                        Nu        Expected Range   Pass?
----------------------------------------------------------------------------------
external.pohlhausen_plate                  182.92  [ 150.0,  220.0]       ✓



## Section 3 — Tube Bank (Staggered)

Air at 300K through a staggered bank: D=25mm, S_T=37.5mm, S_L=31.25mm, U=6 m/s.
Re≈9 570. Reference: Incropera 7th §7.6.

In [5]:
air_bank = EngineeringState(
    fluid=air_props,
    geometry=Geometry(
        geometry_type="tube_bank",
        characteristic_length=0.025,
        pitch_transverse=0.0375,
        pitch_longitudinal=0.03125,
        arrangement="staggered",
    ),
    boundary=BoundaryConditions(boundary_type="constant_wall_temperature"),
    flow=FlowState(velocity=6.0),
)
print(f"Tube bank → Re={air_bank.reynolds:.0f}, Pr={air_bank.prandtl:.3f}")
rows_bank = run_case("air_tube_bank", air_bank, nu_lo=70, nu_hi=130)
print_case("Air 300K — Staggered Tube Bank", rows_bank)

Tube bank → Re=9564, Pr=0.707
--- Air 300K — Staggered Tube Bank ---
 Method                                        Nu        Expected Range   Pass?
----------------------------------------------------------------------------------
tube_banks.grimison                         78.35  [  70.0,  130.0]       ✓
tube_banks.zukauskas                        78.35  [  70.0,  130.0]       ✓



## Section 4 — Summary: All Cases, All Methods

The chart below shows Nu from every applicable correlation per case.
The shaded band indicates the textbook-derived expected range.
Green bars are within range; red bars fall outside.

In [6]:
all_rows = rows_water + rows_air_tube + rows_cylinder + rows_plate + rows_bank

cases_order = [
    "water_60C",
    "air_300K_tube",
    "air_cylinder",
    "air_flat_plate",
    "air_tube_bank",
]
case_titles = {
    "water_60C": "Water 60°C<br>(tube)",
    "air_300K_tube": "Air 300K<br>(tube)",
    "air_cylinder": "Air 300K<br>(cylinder)",
    "air_flat_plate": "Air 300K<br>(flat plate)",
    "air_tube_bank": "Air 300K<br>(tube bank)",
}

fig = make_subplots(
    rows=1,
    cols=len(cases_order),
    subplot_titles=[case_titles[c] for c in cases_order],
    shared_yaxes=False,
)

for col, case_key in enumerate(cases_order, 1):
    rows = [r for r in all_rows if r["case"] == case_key]
    if not rows:
        continue
    methods = [r["method"].split(".")[-1] for r in rows]
    nus = [r["nu"] for r in rows]
    colors = ["#2ecc71" if r["pass_"] else "#e74c3c" for r in rows]
    nu_lo, nu_hi = rows[0]["nu_lo"], rows[0]["nu_hi"]

    fig.add_trace(
        go.Bar(
            y=methods,
            x=nus,
            orientation="h",
            marker_color=colors,
            showlegend=False,
            hovertemplate="%{y}: Nu=%{x:.1f}<extra></extra>",
        ),
        row=1,
        col=col,
    )
    fig.add_vrect(
        x0=nu_lo,
        x1=nu_hi,
        fillcolor="rgba(46,204,113,0.15)",
        line_width=1,
        line_color="#27ae60",
        annotation_text=f"[{nu_lo},{nu_hi}]",
        annotation_position="top right",
        row=1,
        col=col,
    )

fig.update_layout(
    title_text="<b>htcie Benchmark: Nu per Correlation vs. Textbook Range</b>",
    height=420,
    margin=dict(t=80, b=40, l=40, r=20),
)
fig.show()

total = len(all_rows)
passed = sum(1 for r in all_rows if r["pass_"])
print(f"\nSummary: {passed}/{total} method-case combinations within textbook range.")


Summary: 14/14 method-case combinations within textbook range.
